### 5장 5강: 맥락 압축·검색 최적화
**학습 목표**

- 긴 문서에서 불필요한 노이즈를 걸러내고 질문과 관련된 핵심 문장만 압축할 수 있다.
- LLM 기반 압축기를 기본 검색기와 결합하여 ContextualCompressionRetriever를 구현할 수 있다.
- 압축된 문맥을 LLM 모델과 연결하여 노이즈 없는 근거 기반 질의응답 파이프라인을 완성할 수 있다.

문서 전체 전달 방식의 한계
- **토큰 낭비 및 비용 증가**: 질문과 상관없는 수천 자의 본문이 프롬프트에 들어가면 API 호출 비용이 불필요하게 늘어납니다.
- **Lost in the Middle (중간 정보 누락)**: 인공지능 모델은 제공된 긴 문맥의 중간에 위치한 세부 사실을 망각하는 경향이 있습니다.
- **환각(Hallucination) 유발**: 문서 내에 포함된 다른 주제의 수치나 규정이 모델의 판단을 흐려 오답을 생성할 확률이 높아집니다.

맥락 압축(Contextual Compression)
- 검색된 문서 전체를 그대로 전달하지 않고, 질문과 연관된 핵심 내용만 발췌하거나 불필요한 문장을 제거하여 전달하는 검색 최적화 기술입니다.

맥락 압축의 2단계 동작 원리
- **1단계 (기본 검색)**: 키워드 검색기(BM25) 또는 벡터 검색기가 질문과 관련된 후보 문서를 빠르게 가져옵니다.
- **2단계 (문맥 압축)**: 문서 압축기(Compressor)가 질문을 기준으로 문서를 읽고, 무관한 문장을 삭제하거나 핵심 문장만 추출합니다.

텍스트 데이터 준비 및 기본 검색기 생성

In [1]:
# 문서 객체와 기본 키워드 검색기 모듈을 가져옵니다.
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# 여러 주제가 혼합된 실습용 사내 규정 문서를 정의합니다.
raw_documents = [
    Document(
        page_content="복지 제도 안내: 사내 복지 포인트는 매년 1월 1일에 100만 포인트가 지급됩니다. "
                     "포인트는 자기계발 및 문화생활에 사용 가능하며, 미사용 잔여 포인트는 당해 연도 12월 31일에 전액 소멸되어 다음 해로 이월되지 않습니다."
    ),
    Document(
        page_content="휴가 규정 안내: 연차 유급휴가는 1년간 80% 이상 출근 시 15일이 부여됩니다. "
                     "경조사 휴가는 사유 발생일로부터 사용해야 하며, 미사용 연차는 연말에 수당으로 정산 지급됩니다."
    ),
    Document(
        page_content="근무 환경 가이드: 사무실 내 회의실 예약은 사내 인트라넷 시스템을 통해 최소 1시간 전에 신청해야 합니다. "
                     "점심시간은 12시 30분부터 13시 30분까지이며 부서별 자율 운영이 가능합니다."
    )
]

# 공백 기준 단순 토크나이저 함수를 정의합니다.
def custom_tokenizer(text: str) -> list[str]:
    # 문장을 띄어쓰기 단위로 분리하여 단어 리스트를 반환합니다.
    return text.split()

# 1단계 기본 검색기(BM25)를 생성하고 검색 개수를 1개로 설정합니다.
base_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer
)
base_retriever.k = 2

/var/folders/5s/z1p5lnl57_7_ccp6c8lxp7jr0000gn/T/ipykernel_73045/1708698093.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


LLM 모델 기반 문서 압축기 생성

In [5]:
from langchain.chat_models import init_chat_model
# LLM을 활용해 문맥을 오려내는 압축기 모듈을 가져옵니다.
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
# 기본 검색기와 압축기를 결합하는 래퍼 클래스를 가져옵니다.
from langchain_classic.retrievers import ContextualCompressionRetriever

model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="low"
)

compressor = LLMChainExtractor.from_llm(model)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

압축 전후 검색 결과 비교 분석

In [7]:
query = "사내 복지 포인트 남으면 다음 해로 넘겨서 쓸 수 있나요?"

print("=== BM25 ===")
bm25_docs = base_retriever.invoke(query)
for i, doc in enumerate(bm25_docs, 1):
    print(f"순서: {i}번째 Doc: {doc.page_content}")

print("=== Compression ===")
com_docs = compression_retriever.invoke(query)
for i, doc in enumerate(com_docs, 1):
    print(f"순서: {i}번째 Doc: {doc.page_content}")

=== BM25 ===
순서: 1번째 Doc: 복지 제도 안내: 사내 복지 포인트는 매년 1월 1일에 100만 포인트가 지급됩니다. 포인트는 자기계발 및 문화생활에 사용 가능하며, 미사용 잔여 포인트는 당해 연도 12월 31일에 전액 소멸되어 다음 해로 이월되지 않습니다.
순서: 2번째 Doc: 근무 환경 가이드: 사무실 내 회의실 예약은 사내 인트라넷 시스템을 통해 최소 1시간 전에 신청해야 합니다. 점심시간은 12시 30분부터 13시 30분까지이며 부서별 자율 운영이 가능합니다.
=== Compression ===
순서: 1번째 Doc: 사내 복지 포인트는 매년 1월 1일에 100만 포인트가 지급됩니다. 포인트는 자기계발 및 문화생활에 사용 가능하며, 미사용 잔여 포인트는 당해 연도 12월 31일에 전액 소멸되어 다음 해로 이월되지 않습니다.


LM 모델 연동 질의응답 파이프라인 완성

In [8]:
# 프롬프트 템플릿 모듈을 가져옵니다.
from langchain_core.prompts import PromptTemplate

# 문맥과 질문을 주입할 프롬프트 템플릿을 생성합니다.
qa_prompt = PromptTemplate.from_template(
    "다음 제공된 [참고 문서]를 바탕으로 [질문]에 대해 명확하고 간결하게 답변하세요.\n\n"
    "[참고 문서]:\n{context}\n\n"
    "[질문]:\n{question}\n\n"
    "[답변]:"
)

# 압축 검색기를 통해 핵심 문맥을 추출합니다.
compressed_docs = compression_retriever.invoke(query)
context_text = "\n".join([doc.page_content for doc in compressed_docs])

# 프롬프트와 gpt-5-mini 모델을 연결해 질의응답 체인을 구성합니다.
qa_chain = qa_prompt | model

# 체인을 실행하고 결과를 출력합니다.
response = qa_chain.invoke({
    "context": context_text,
    "question": query
})

print(f"[질문]: {query}\n")
print("[생성 답변]")
print(response.content)

[질문]: 사내 복지 포인트 남으면 다음 해로 넘겨서 쓸 수 있나요?

[생성 답변]
아니요. 미사용 복지 포인트는 매년 12월 31일에 전액 소멸되어 다음 해로 이월되지 않습니다.
